# 02 — Espina: edad × manzana como restricción dura

Parte C de `microdatos.ipynb`, migrada al módulo `manzanas/`. Comuna de trabajo: Independencia (`COMUNA_CUT = 13108`).

No depende del kernel de otro notebook: lee la comuna desde `resultados/` (la primera vez la prepara desde `databases/`, ~1 min) y al final guarda la espina en `resultados/espina_13108.npz`, que usan los notebooks 03 y 04. C0 recalcula los insumos de la prueba de concepto (`01_poc_historica.ipynb`) y verifica que se reproduzcan sus números.

In [1]:
import os

import numpy as np
import pandas as pd

from manzanas.datos import cargar_comuna
from manzanas.inventario import EDAD_TRAMOS
from manzanas.ipf import ipf, sinkhorn
from manzanas.validacion import errores_marginales, srmse

COMUNA_CUT = 13108
datos = cargar_comuna(COMUNA_CUT)
nu, manzanas_ids = datos.nu, datos.manzanas_ids
print("personas:", len(datos.personas), "| manzanas con población:", len(nu), "| suma n_per:", nu.sum())
assert nu.sum() == len(datos.personas)

personas: 116943 | manzanas con población: 469 | suma n_per: 116943


### C0. Insumos de la prueba de concepto (recalculados) y regresión

Con el módulo se recalculan:
- μ y ν de edad, el costo con `prom_edad` y Sinkhorn con ε = 1,2;
- la semilla informada con sexo;
- el marginal de inmigrantes con sus `*`;
- el modelo de 3 variables de B6.

Deben reproducir la cadena de SRMSE de la edad: 0,794 → 0,538 → 0,539 → 0,345.

In [2]:
# C0. Insumos de las Partes 0-B, recalculados con el módulo
edad_np = datos.personas["edad"].to_numpy()
sexo_np = datos.personas["sexo"].to_numpy()  # 1 = hombre, 2 = mujer
es_inmigrante = (datos.personas["p25_lug_nacimiento_rec"].to_numpy() == 2).astype(int)  # el -99 queda con no inmigrante

# Parte 0: edad comunal (mu), n_per (nu) y edad publicada por manzana (Y_real)
mu = np.array([int(((edad_np >= lo) & (edad_np <= hi)).sum()) for _, lo, hi in EDAD_TRAMOS])
Y_real = datos.manzanas[[f"n_edad_{n}" for n, _, _ in EDAD_TRAMOS]].to_numpy().T.astype(int)
assert mu.sum() == nu.sum() == len(edad_np)
srmse_base = srmse(np.outer(mu, nu) / mu.sum(), Y_real)

prom_edad_com = datos.manzanas["prom_edad"].astype(str).str.replace(",", ".", regex=False).astype(float).to_numpy()
prom_edad_com = np.nan_to_num(prom_edad_com, nan=0)
midpoints = np.array([edad_np[(edad_np >= lo) & (edad_np <= hi)].mean() for _, lo, hi in EDAD_TRAMOS])
C = (midpoints[:, None] - prom_edad_com[None, :]) ** 2
C = C / C.std()
counts_best = sinkhorn(mu, nu, C, 1.2, n_iter=20000, tol=1e-10) * mu.sum()
srmse_edad = srmse(counts_best, Y_real)

# Parte A: sexo, con la semilla informada (Sinkhorn x reparto por sexo de cada manzana)
mu_edad_sexo = np.zeros((len(EDAD_TRAMOS), 2), dtype=int)
for i, (_, lo, hi) in enumerate(EDAD_TRAMOS):
    for s in [1, 2]:
        mu_edad_sexo[i, s - 1] = int(((edad_np >= lo) & (edad_np <= hi) & (sexo_np == s)).sum())
nu_sexo = np.vstack([datos.manzanas["n_hombres"].to_numpy(), datos.manzanas["n_mujeres"].to_numpy()]).astype(int)
seed = counts_best[:, None, :] * (nu_sexo / nu_sexo.sum(axis=0, keepdims=True))[None, :, :]
T_informado, _, _ = ipf(seed, [((2,), mu_edad_sexo), ((0,), nu_sexo)], n_iter=500, tol=1e-10)
srmse_sexo = srmse(T_informado.sum(axis=1), Y_real)

# Parte B: inmigrante, con NaN donde el INE puso '*'
n_inmigrantes_manzana = pd.to_numeric(datos.manzanas["n_inmigrantes"].astype(str), errors="coerce").to_numpy()
mask_enmascarada = np.isnan(n_inmigrantes_manzana)
nu_inmigrante = np.vstack([n_inmigrantes_manzana, nu - n_inmigrantes_manzana])  # fila 0 = inmigrante, 1 = resto
mu_edad_sexo_inmigrante = np.zeros((len(EDAD_TRAMOS), 2, 2), dtype=int)       # último eje: [inmigrante, no_inmigrante]
for i, (_, lo, hi) in enumerate(EDAD_TRAMOS):
    for s in [1, 2]:
        base = (edad_np >= lo) & (edad_np <= hi) & (sexo_np == s)
        mu_edad_sexo_inmigrante[i, s - 1, 0] = int((base & (es_inmigrante == 1)).sum())
        mu_edad_sexo_inmigrante[i, s - 1, 1] = int((base & (es_inmigrante == 0)).sum())
tasa = es_inmigrante.mean()
seed_4d = T_informado[:, :, None, :] * np.array([tasa, 1 - tasa])[None, None, :, None]
constraints_4d = [((3,), mu_edad_sexo_inmigrante), ((0, 2), nu_sexo), ((0, 1), nu_inmigrante)]
T_4d, n_it, convergio = ipf(seed_4d, constraints_4d, n_iter=50000, tol=1e-10)
srmse_inm = srmse(T_4d.sum(axis=(1, 2)), Y_real)

print(f"manzanas con n_inmigrantes enmascarado: {mask_enmascarada.sum()} | B6: {n_it} iteraciones, convergió={convergio}")
print(f"SRMSE edad: baseline={srmse_base:.4f} | +prom_edad={srmse_edad:.4f} | +sexo={srmse_sexo:.4f} | +inmigrante={srmse_inm:.4f}")
for esperado, obtenido in [(0.7940, srmse_base), (0.5382, srmse_edad), (0.5393, srmse_sexo), (0.3452, srmse_inm)]:
    assert abs(obtenido - esperado) < 5e-4, (esperado, obtenido)
assert convergio

manzanas con n_inmigrantes enmascarado: 45 | B6: 5354 iteraciones, convergió=True
SRMSE edad: baseline=0.7940 | +prom_edad=0.5382 | +sexo=0.5393 | +inmigrante=0.3452


# Parte C — Escalamiento, paso 0: espina con edad × manzana exacta

Hasta ahora los conteos `n_edad_*` por manzana ($Y_{real}$) quedaban fuera del ajuste para usarlos como *ground truth* de la prueba de concepto. Como son un marginal publicado, en el modelo final deben entrar como **restricción dura**. La **espina** del modelo pasa a ser el tensor

$$
T[e,s,i,m],\qquad e\in\{7\text{ tramos}\},\; s\in\{H,M\},\; i\in\{\text{inmigrante},\text{no inmigrante}\},\; m\in\{469\text{ manzanas}\}
$$

sujeto a cuatro restricciones:

$$
\sum_{m} T = \mu[e,s,i],\qquad
\sum_{s,i} T = \nu^{E}[e,m],\qquad
\sum_{e,i} T = \nu^{S}[s,m],\qquad
\sum_{e,s} T = \nu^{I}[i,m]\;\;(\text{solo manzanas no enmascaradas}).
$$

En Independencia `n_edad_*` no tiene celdas enmascaradas, cada manzana suma `n_per` y cada tramo suma el total comunal (C1). El tensor real cumple todas las restricciones, así que el problema es factible y la proyección KL existe y es única (Csiszár, 1975).

**Consecuencia 1: la semilla con `prom_edad` deja de importar.** IPF desde una semilla uniforme entrega el estimador de máxima verosimilitud del modelo log-lineal jerárquico $[ESI][EM][SM][IM]$:

$$
\log T^*[e,s,i,m] = \lambda_{ESI}[e,s,i] + \lambda_{EM}[e,m] + \lambda_{SM}[s,m] + \lambda_{IM}[i,m].
$$

Si $\log T^{(0)}$ también es una suma de términos sobre esos mismos pares, la proyección llega exactamente al mismo $T^*$. Ese es el caso de `seed_4d`: $e^{-C_{em}/\varepsilon}$ es un término $EM$, la proporción de sexo por manzana es un término $SM$ y la tasa comunal de inmigración es un término $I$. Por lo tanto, una vez que $E\times M$ es conocido, el costo basado en `prom_edad` (la palanca principal hasta ahora) deja de aportar (C4). Es la lección 4 generalizada: una semilla solo informa a través de interacciones que las restricciones **no** fijan, por ejemplo $E\times S\times M$.

**Consecuencia 2: $Y_{real}$ ya no sirve para validar**, porque su SRMSE es 0 por construcción. A partir de ahora validamos con *leave-one-out* (LOO): se quita un marginal publicado del ajuste, se predice con el resto y se compara contra el dato.
- **LOO interno (esta tanda, C5):** se quita $\nu^{S}$ o $\nu^{I}$ y se compara con y sin $E\times M$ dura. Quitar $\nu^{E}$ con la semilla de `prom_edad` es exactamente el modelo de B6 y debe reproducir 0.345 (chequeo de regresión).
- **LOO externo (siguiente tanda):** `n_ocupado` y `n_estcivcon_*`, variables que no están en la espina, predichas con la extensión cerrada del nivel 2.

**Qué no identifica la espina:** las interacciones entre la manzana y dos o más atributos a la vez, como la variación de la razón hombre/mujer *dentro de un tramo* entre manzanas. Esas salen completas de $\mu$; el modelo supone que, dados los marginales de cada manzana, la estructura conjunta es la comunal.

### C1. `n_edad_*` como restricción: consistencia entre fuentes

In [3]:
# C1. n_edad_* deja de ser ground truth y pasa a ser restricción dura: chequeos de consistencia
nu_edad = Y_real.astype(float)  # (7, 469), mismo orden de manzanas que nu / manzanas_ids
mu_edad_desde_esi = mu_edad_sexo_inmigrante.sum(axis=(1, 2))

print("forma nu_edad:", nu_edad.shape, "| NaN:", np.isnan(nu_edad).sum())
print("suma por tramo, manzanas:", nu_edad.sum(axis=1).astype(int))
print("suma por tramo, comuna  :", mu_edad_desde_esi)
print("max |suma por manzana - n_per|:", np.abs(nu_edad.sum(axis=0) - nu).max())

assert not np.isnan(nu_edad).any()
assert (nu_edad.sum(axis=0) == nu).all()                 # cada manzana: suma de tramos = n_per
assert (nu_edad.sum(axis=1) == mu_edad_desde_esi).all()  # cada tramo: suma de manzanas = total comunal
assert (mu_edad_desde_esi == mu).all()
print("OK: edad x manzana es consistente con n_per y con la conjunta comunal")

forma nu_edad: (7, 469) | NaN: 0
suma por tramo, manzanas: [ 7554 11542  5280 11439 44178 19274 17676]
suma por tramo, comuna  : [ 7554 11542  5280 11439 44178 19274 17676]
max |suma por manzana - n_per|: 0.0
OK: edad x manzana es consistente con n_per y con la conjunta comunal


### C2. Utilidades de validación

`srmse` y `errores_marginales` están ahora en `manzanas/validacion.py`. Esta celda verifica que reproduzcan los números de la Parte B.

In [4]:
# C2. Regresión de las utilidades del módulo: las utilidades deben reproducir los números ya validados de la Parte B
print("SRMSE edad, T_4d (esperado 0.3452):", round(srmse(T_4d.sum(axis=(1, 2)), Y_real), 4))
for k, v in errores_marginales(T_4d, constraints_4d, ["mu(e,s,i)", "nu(s,m)", "nu(i,m)"]).items():
    print(f"  T_4d, error marginal {k}: {v:.2e}")

SRMSE edad, T_4d (esperado 0.3452): 0.3452
  T_4d, error marginal mu(e,s,i): 6.43e-07
  T_4d, error marginal nu(s,m): 1.19e-09
  T_4d, error marginal nu(i,m): 1.36e-12


### C3. Ajuste de la espina con 4 restricciones

Semilla uniforme: por el argumento log-lineal de arriba, cualquier semilla con solo términos $ESI$, $EM$, $SM$ o $IM$ da el mismo resultado. Las 45 manzanas con `n_inmigrantes` enmascarado deben absorber exactamente el residuo comunal de 98 inmigrantes.

In [5]:
# C3. Espina T[edad, sexo, inmigrante, manzana] con edad x manzana como restricción dura
constraints_espina = [
    ((3,),   mu_edad_sexo_inmigrante),  # conjunta comunal (7,2,2)
    ((1, 2), nu_edad),                  # edad x manzana (7,469)  <- nueva
    ((0, 2), nu_sexo),                  # sexo x manzana (2,469)
    ((0, 1), nu_inmigrante),            # inmigrante x manzana (2,469), NaN en las 45 enmascaradas
]
nombres_espina = ["mu(e,s,i)", "nu(e,m)", "nu(s,m)", "nu(i,m)"]

T_espina, n_it, convergio = ipf(np.ones(seed_4d.shape), constraints_espina, n_iter=100000, tol=1e-10)
print("iteraciones:", n_it, "| convergió:", convergio, "| forma:", T_espina.shape)
errs_espina = errores_marginales(T_espina, constraints_espina, nombres_espina)
for k, v in errs_espina.items():
    print(f"  error marginal {k}: {v:.2e}")

# Las manzanas enmascaradas deben absorber exactamente el residuo comunal de inmigrantes
residuo_inm = mu_edad_sexo_inmigrante[:, :, 0].sum() - np.nansum(nu_inmigrante[0])
inm_por_manzana_masc = T_espina[:, :, 0, mask_enmascarada].sum(axis=(0, 1))  # (45,)
print(f"inmigrantes estimados en manzanas enmascaradas: {inm_por_manzana_masc.sum():.6f} | residuo esperado: {residuo_inm}")
print(f"  por manzana enmascarada: min={inm_por_manzana_masc.min():.2f}, max={inm_por_manzana_masc.max():.2f}")
print("min T_espina:", T_espina.min(), "| total:", T_espina.sum())

# Y_real ya no valida nada: la edad implícita reproduce el dato por construcción
print("SRMSE edad implícita vs Y_real (debe ser ~0):", srmse(T_espina.sum(axis=(1, 2)), Y_real))

assert convergio
assert all(v < 1e-5 for v in errs_espina.values())  # lo esperado es <~1e-6; el assert deja margen
assert abs(inm_por_manzana_masc.sum() - residuo_inm) < 1e-4
assert abs(T_espina.sum() - len(datos.personas)) < 1e-6

iteraciones: 5242 | convergió: True | forma: (7, 2, 2, 469)
  error marginal mu(e,s,i): 6.58e-07
  error marginal nu(e,m): 2.81e-08
  error marginal nu(s,m): 1.34e-09
  error marginal nu(i,m): 4.55e-13
inmigrantes estimados en manzanas enmascaradas: 98.000003 | residuo esperado: 98.0
  por manzana enmascarada: min=0.32, max=4.72
min T_espina: 0.0 | total: 116943.0
SRMSE edad implícita vs Y_real (debe ser ~0): 4.136302048881838e-11


### C4. Con $E\times M$ dura, la semilla de `prom_edad` es inerte

Se ajusta la misma espina desde `seed_4d` (Sinkhorn con `prom_edad` × sexo por manzana × tasa comunal de inmigración). Si el argumento log-lineal es correcto, la diferencia con C3 debe ser del orden de la tolerancia. Como contraste, sin la restricción $E\times M$ (modelo de la Parte B) la semilla sí cambia el resultado.

In [6]:
# C4. Misma espina desde seed_4d: debe dar el mismo T que la semilla uniforme
T_espina_s4d, n_it_s4d, conv_s4d = ipf(seed_4d, constraints_espina, n_iter=100000, tol=1e-10)
dif_abs = np.abs(T_espina - T_espina_s4d)
dif_rel = np.divide(dif_abs, T_espina, out=np.zeros_like(dif_abs), where=T_espina > 1)
print("semilla seed_4d -> iteraciones:", n_it_s4d, "| convergió:", conv_s4d)
print("max |T_espina(uniforme) - T_espina(seed_4d)|:", dif_abs.max())
print("max diferencia relativa (celdas > 1 persona):", dif_rel.max())

# Contraste: sin E x M (restricciones de la Parte B) la semilla sí importa
T_4d_unif, n_it_u, conv_u = ipf(np.ones(seed_4d.shape), constraints_4d, n_iter=100000, tol=1e-10)
print("\nsin E x M, semilla uniforme -> iteraciones:", n_it_u, "| convergió:", conv_u)
print("  SRMSE edad con semilla uniforme:", round(srmse(T_4d_unif.sum(axis=(1, 2)), Y_real), 4),
      "| con seed_4d (T_4d):", round(srmse(T_4d.sum(axis=(1, 2)), Y_real), 4))
print("  max |T_4d - T_4d(uniforme)|:", np.abs(T_4d - T_4d_unif).max())

assert conv_s4d and conv_u
assert dif_abs.max() < 1e-4

semilla seed_4d -> iteraciones: 5242 | convergió: True
max |T_espina(uniforme) - T_espina(seed_4d)|: 5.5706550483591855e-12
max diferencia relativa (celdas > 1 persona): 5.369443289213964e-13

sin E x M, semilla uniforme -> iteraciones: 5358 | convergió: True
  SRMSE edad con semilla uniforme: 0.5328 | con seed_4d (T_4d): 0.3452
  max |T_4d - T_4d(uniforme)|: 144.29654422556132


### C5. Leave-one-out interno

Para cada marginal de manzana $k\in\{E,S,I\}$ se ajusta sin $\nu^{k}$ y se predice $\hat\nu^{k}=\operatorname{proj}_k(T)$. Se comparan tres modelos:

1. **Proporcional:** composición comunal del atributo × `n_per` de la manzana.
2. **Sin $E\times M$:** restricciones de la Parte B, con la semilla de `prom_edad`.
3. **Con $E\times M$ dura:** la espina sin $\nu^{k}$.

La semilla de (2) es `counts_best` (solo usa `prom_edad` y `n_per`) y no `seed_4d`: `seed_4d` ya contiene $\nu^{S}$ a través de `T_informado` y filtraría el dato en el LOO de sexo. En el LOO de edad ambas semillas dan el mismo $T$ por el argumento de C4, así que esa fila debe reproducir 0.794 y 0.345. En inmigrante, el SRMSE se calcula solo sobre las manzanas no enmascaradas.

In [7]:
# C5. Leave-one-out interno: se quita un marginal de manzana, se ajusta con el resto y se predice el quitado
N = mu_edad_sexo_inmigrante.sum()
c_mu, c_edad, c_sexo, c_inm = constraints_espina

# Semilla "sin E x M": solo prom_edad y n_per (Sinkhorn de la Parte 0); no filtra sexo ni inmigrante
seed_prom_edad = np.broadcast_to(counts_best[:, None, None, :], seed_4d.shape)
seed_unif = np.ones(seed_4d.shape)

def proporcional(axes):
    # Baseline: composición comunal del atributo repartida según n_per de cada manzana
    share = mu_edad_sexo_inmigrante.sum(axis=axes) / N
    return share[:, None] * nu[None, :]

casos = [
    # (marginal quitado, dato publicado, ejes a sumar para predecirlo, restricciones sin E x M, con E x M)
    ("edad",       Y_real,        (1, 2), [c_mu, c_sexo, c_inm], None),
    ("sexo",       nu_sexo,       (0, 2), [c_mu, c_inm],         [c_mu, c_edad, c_inm]),
    ("inmigrante", nu_inmigrante, (0, 1), [c_mu, c_sexo],        [c_mu, c_edad, c_sexo]),
]

filas = []
for nombre, real, axes, ctr_sin, ctr_con in casos:
    fila = {"marginal quitado": nombre, "proporcional": srmse(proporcional(axes), real)}
    for etiqueta, seed_k, ctr in [("sin E×M (semilla prom_edad)", seed_prom_edad, ctr_sin),
                                  ("con E×M dura", seed_unif, ctr_con)]:
        if ctr is None:
            continue
        T_k, it_k, conv_k = ipf(seed_k, ctr, n_iter=100000, tol=1e-10)
        err_k = max(errores_marginales(T_k, ctr).values())
        print(f"[{nombre:>10} | {etiqueta:<27}] iteraciones={it_k:>6} | convergió={conv_k} | max error marginal={err_k:.2e}")
        assert conv_k
        fila[etiqueta] = srmse(T_k.sum(axis=axes), real)
    filas.append(fila)

tabla_loo = pd.DataFrame(filas).set_index("marginal quitado")
print("\nSRMSE del marginal quitado (inmigrante: solo manzanas no enmascaradas)")
print(tabla_loo.round(4).to_string())

# Regresión: quitar edad reproduce el baseline (0.7940) y el modelo B6 (0.3452)
assert abs(tabla_loo.loc["edad", "proporcional"] - 0.7940) < 5e-4
assert abs(tabla_loo.loc["edad", "sin E×M (semilla prom_edad)"] - 0.3452) < 5e-4

[      edad | sin E×M (semilla prom_edad)] iteraciones=  5354 | convergió=True | max error marginal=6.43e-07
[      sexo | sin E×M (semilla prom_edad)] iteraciones=  8896 | convergió=True | max error marginal=1.17e-06
[      sexo | con E×M dura               ] iteraciones=  5234 | convergió=True | max error marginal=6.71e-07
[inmigrante | sin E×M (semilla prom_edad)] iteraciones=    11 | convergió=True | max error marginal=2.67e-10
[inmigrante | con E×M dura               ] iteraciones=    13 | convergió=True | max error marginal=6.18e-11

SRMSE del marginal quitado (inmigrante: solo manzanas no enmascaradas)
                  proporcional  sin E×M (semilla prom_edad)  con E×M dura
marginal quitado                                                         
edad                    0.7940                       0.3452           NaN
sexo                    0.0744                       0.0797        0.0797
inmigrante              0.9268                       0.8688        0.7813


### C6. Guardar la espina

`T_espina`, los marginales de la espina y el LOO interno quedan en `resultados/` para los notebooks 03 y 04.

In [8]:
# C6. Guardar la espina y el LOO interno
ruta_espina = os.path.join("resultados", f"espina_{COMUNA_CUT}.npz")
np.savez(ruta_espina, T_espina=T_espina, mu_edad_sexo_inmigrante=mu_edad_sexo_inmigrante, nu_edad=nu_edad,
         nu_sexo=nu_sexo, nu_inmigrante=nu_inmigrante, mask_enmascarada=mask_enmascarada,
         loo_inmigrante_proporcional=tabla_loo.loc["inmigrante", "proporcional"],
         loo_inmigrante_con_EM=tabla_loo.loc["inmigrante", "con E×M dura"])
tabla_loo.to_csv(os.path.join("resultados", f"loo_interno_{COMUNA_CUT}.csv"))
print("guardado:", ruta_espina, f"({os.path.getsize(ruta_espina) / 1e6:.2f} MB)")

guardado: resultados/espina_13108.npz (0.15 MB)
